# LeetCode 709 — To Lower Case bằng Python

**Link đề:** https://leetcode.com/problems/to-lower-case/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def toLowerCase(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu **bảng mã ASCII**: mỗi ký tự là một con số; `ord()` và `chr()` chuyển qua lại.
2. Phát hiện quy luật: chữ thường = chữ hoa **+ 32**, và 32 chính là **một bit** (bit thứ 5).
3. Hiểu chuỗi Python **bất biến** (immutable) → vì sao nên gom ký tự vào list rồi `''.join(...)`.
4. Cài 5 cách: `str.lower()` → `ord/chr + 32` → phép OR bit `| 32` → bảng tra `dict` → `str.translate`.
5. Trace, chứng minh đúng, Time/Space, các lỗi kinh điển (đổi cả ký tự không phải chữ hoa) và biến thể (to upper, đảo hoa/thường, Caesar).

## 1. Đề bài và ví dụ

Cho chuỗi `s`. Trả về chuỗi sau khi thay **mọi chữ cái hoa** bằng **chữ thường tương ứng**. Các ký tự khác giữ nguyên.

| `s` | Output |
|---|---|
| `"Hello"` | `"hello"` |
| `"here"` | `"here"` |
| `"LOVELY"` | `"lovely"` |

**Constraints:** `1 <= len(s) <= 100`; `s` gồm các ký tự ASCII in được (printable): chữ, số, dấu câu, khoảng trắng... (mã 32 đến 126).

**Hint của đề:** Hầu hết ngôn ngữ đã có hàm `lower`, nhưng mục đích bài là **tự cài** nó. *Think ASCII!*

In [1]:
for s in ["Hello", "here", "LOVELY"]:
    print(repr(s), "->", repr(s.lower()))

'Hello' -> 'hello'
'here' -> 'here'
'LOVELY' -> 'lovely'


## 2. Nền tảng: ký tự là con số — `ord` và `chr`

- `ord(ch)`: trả **mã số** (code point) của ký tự `ch`.
- `chr(n)`: trả ký tự có mã `n`.

Với ASCII, mã nằm trong `0..127`.

In [2]:
print(ord('A'), ord('B'), ord('Z'))
print(ord('a'), ord('b'), ord('z'))
print(chr(72), chr(104), chr(48))

65 66 90
97 98 122
H h 0


### 2.1. Bảng ASCII in được (32..126)

In ra để **tự thấy** quy luật: chữ hoa `A..Z` là `65..90`, chữ thường `a..z` là `97..122`.

In [3]:
for start in range(32, 127, 16):
    row = "  ".join(f"{n:3}:{chr(n)}" for n in range(start, min(start + 16, 127)))
    print(row)

 32:    33:!   34:"   35:#   36:$   37:%   38:&   39:'   40:(   41:)   42:*   43:+   44:,   45:-   46:.   47:/
 48:0   49:1   50:2   51:3   52:4   53:5   54:6   55:7   56:8   57:9   58::   59:;   60:<   61:=   62:>   63:?
 64:@   65:A   66:B   67:C   68:D   69:E   70:F   71:G   72:H   73:I   74:J   75:K   76:L   77:M   78:N   79:O
 80:P   81:Q   82:R   83:S   84:T   85:U   86:V   87:W   88:X   89:Y   90:Z   91:[   92:\   93:]   94:^   95:_
 96:`   97:a   98:b   99:c  100:d  101:e  102:f  103:g  104:h  105:i  106:j  107:k  108:l  109:m  110:n  111:o
112:p  113:q  114:r  115:s  116:t  117:u  118:v  119:w  120:x  121:y  122:z  123:{  124:|  125:}  126:~


### 2.2. Khoảng cách chữ hoa – chữ thường

`ord('a') - ord('A') = 97 - 65 = 32`. Kiểm tra với **mọi** chữ cái:

In [4]:
diffs = {ord(chr(c).lower()) - c for c in range(ord('A'), ord('Z') + 1)}
print("Mọi hiệu ord(thường) - ord(hoa):", diffs)
print("'H' ->", ord('H'), "+ 32 =", ord('H') + 32, "->", chr(ord('H') + 32))

Mọi hiệu ord(thường) - ord(hoa): {32}
'H' -> 72 + 32 = 104 -> h


### 2.3. So sánh ký tự trực tiếp

Python so sánh chuỗi một ký tự theo mã số, nên `'A' <= ch <= 'Z'` kiểm tra "ch là chữ hoa" mà không cần `ord`.

In [5]:
for ch in ['A', 'M', 'Z', 'a', '[', '@', '5']:
    print(ch, ord(ch), "chữ hoa?", 'A' <= ch <= 'Z')

A 65 chữ hoa? True
M 77 chữ hoa? True
Z 90 chữ hoa? True
a 97 chữ hoa? False
[ 91 chữ hoa? False
@ 64 chữ hoa? False
5 53 chữ hoa? False


Chú ý `'@'` (64) đứng ngay trước `'A'`, `'['` (91) ngay sau `'Z'`: chúng **không** phải chữ hoa nên không được đổi.

### 2.4. Chuỗi bất biến (immutable) và `''.join`

Không gán được `s[0] = 'h'`. Cách chuẩn: gom ký tự kết quả vào **list**, cuối cùng ghép bằng `''.join(list)` (một lần, `O(n)`).

In [6]:
s = "Hello"
try:
    s[0] = 'h'
except TypeError as e:
    print("Lỗi:", e)

parts = ['h', 'e', 'l', 'l', 'o']
print("".join(parts), "|", "-".join(parts))

Lỗi: 'str' object does not support item assignment
hello | h-e-l-l-o


Cộng chuỗi `res += ch` trong vòng lặp cũng chạy được, nhưng về lý thuyết mỗi lần cộng có thể tạo chuỗi mới (`O(n²)` tổng). CPython có tối ưu cho trường hợp này, nhưng thói quen tốt là dùng list + `join`.

## 3. Cách 1 — Hàm có sẵn `str.lower()`

Một dòng, chắc chắn đúng. Nhưng **không đạt mục tiêu học** của bài (hint đã nói rõ). Ta dùng nó làm **oracle** (đáp án chuẩn) để kiểm thử các cách tự cài.

In [7]:
def lower_builtin(s: str) -> str:
    return s.lower()

print(lower_builtin("Hello, World 123!"))

hello, world 123!


## 4. Cách 2 — Duyệt từng ký tự, dùng `ord/chr + 32` (lời giải chính)

Mổ xẻ:
1. Tạo list rỗng `res`.
2. Với mỗi `ch` trong `s`:
   - nếu `'A' <= ch <= 'Z'` → thêm `chr(ord(ch) + 32)`;
   - ngược lại → thêm nguyên `ch`.
3. Trả `''.join(res)`.

In [8]:
def lower_ascii(s: str) -> str:
    res = []
    for ch in s:
        if 'A' <= ch <= 'Z':
            res.append(chr(ord(ch) + 32))
        else:
            res.append(ch)
    return "".join(res)

for s in ["Hello", "here", "LOVELY", "al&phaBET", "@[Z]` 9"]:
    print(repr(s), "->", repr(lower_ascii(s)))

'Hello' -> 'hello'
'here' -> 'here'
'LOVELY' -> 'lovely'
'al&phaBET' -> 'al&phabet'
'@[Z]` 9' -> '@[z]` 9'


## 5. TRACE từng bước — `s = "Hi@Z!"`

| i | ch | ord(ch) | Chữ hoa? | Ký tự thêm vào | res |
|---|---|:---:|:---:|---|---|
| 0 | H | 72 | có | chr(104) = h | ['h'] |
| 1 | i | 105 | không | i | ['h','i'] |
| 2 | @ | 64 | không (64 < 65) | @ | ['h','i','@'] |
| 3 | Z | 90 | có | chr(122) = z | [...,'z'] |
| 4 | ! | 33 | không | ! | [...,'!'] |

Kết quả: `"hi@z!"`.

In [9]:
def trace_lower(s: str) -> str:
    res = []
    for i, ch in enumerate(s):
        if 'A' <= ch <= 'Z':
            new = chr(ord(ch) + 32)
            note = f"chữ hoa: {ord(ch)} + 32 = {ord(ch) + 32}"
        else:
            new = ch
            note = "giữ nguyên"
        res.append(new)
        print(f"i={i}, ch={ch!r}, ord={ord(ch):3} | {note:24} | thêm {new!r} | res={''.join(res)!r}")
    return "".join(res)

trace_lower("Hi@Z!")

i=0, ch='H', ord= 72 | chữ hoa: 72 + 32 = 104   | thêm 'h' | res='h'
i=1, ch='i', ord=105 | giữ nguyên               | thêm 'i' | res='hi'
i=2, ch='@', ord= 64 | giữ nguyên               | thêm '@' | res='hi@'
i=3, ch='Z', ord= 90 | chữ hoa: 90 + 32 = 122   | thêm 'z' | res='hi@z'
i=4, ch='!', ord= 33 | giữ nguyên               | thêm '!' | res='hi@z!'


'hi@z!'

## 6. Insight sâu hơn — 32 là một bit

Viết mã nhị phân (8 bit):

In [10]:
for ch in "AaZz":
    print(ch, ord(ch), format(ord(ch), "08b"))
print("32 =", format(32, "08b"))

A 65 01000001
a 97 01100001
Z 90 01011010
z 122 01111010
32 = 00100000


`A = 0100 0001`, `a = 0110 0001`: chỉ khác **bit thứ 5** (giá trị 32). Với mọi chữ hoa, bit này đang là `0`. Vậy:
- **Bật** bit 5 bằng phép OR: `ord(ch) | 32` → chữ thường.
- **Tắt** bit 5 bằng AND với phần bù: `ord(ch) & ~32` → chữ hoa.
- **Đảo** bit 5 bằng XOR: `ord(ch) ^ 32` → đổi hoa ↔ thường.

Phép bit:

| Toán tử | Tên | Ví dụ (4 bit) |
|---|---|---|
| `a \| b` | OR: bit = 1 nếu một trong hai là 1 | `0001 \| 0100 = 0101` |
| `a & b` | AND: bit = 1 nếu cả hai là 1 | `0101 & 0100 = 0100` |
| `a ^ b` | XOR: bit = 1 nếu khác nhau | `0101 ^ 0100 = 0001` |
| `~a` | NOT: đảo mọi bit | `~32 = -33` (dạng bù hai) |

In [11]:
print(chr(ord('H') | 32), chr(ord('h') | 32))      # OR: h h (đã thường thì giữ)
print(chr(ord('h') & ~32), chr(ord('H') & ~32))    # AND NOT: H H
print(chr(ord('H') ^ 32), chr(ord('h') ^ 32))      # XOR: h H

h h
H H
h H


## 7. Cách 3 — Phép OR bit

**Chú ý:** vẫn phải kiểm tra `'A' <= ch <= 'Z'`. Nếu OR 32 cho **mọi** ký tự thì ví dụ `'@'` (64) thành `` '`' `` (96), `'['` thành `'{'` — sai!

In [12]:
def lower_bit(s: str) -> str:
    return "".join(chr(ord(ch) | 32) if 'A' <= ch <= 'Z' else ch for ch in s)

print(lower_bit("Hello"), lower_bit("LOVELY"), lower_bit("@[Z]"))

hello lovely @[z]


Cú pháp `"".join(X if điều_kiện else Y for ch in s)`: **generator expression** kết hợp biểu thức điều kiện — sinh từng ký tự rồi `join` ghép lại, không cần list trung gian viết tay.

## 8. Cách 4 — Bảng tra (dict) dựng sẵn

Dựng một `dict` ánh xạ `'A'→'a'`, ..., `'Z'→'z'`. Với mỗi ký tự, `table.get(ch, ch)`: nếu có trong bảng thì lấy chữ thường, không thì giữ nguyên.

In [13]:
UPPER = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
LOWER = "abcdefghijklmnopqrstuvwxyz"
TABLE = dict(zip(UPPER, LOWER))
print(list(TABLE.items())[:5], "...")

def lower_dict(s: str) -> str:
    return "".join(TABLE.get(ch, ch) for ch in s)

print(lower_dict("Hello, WORLD!"))

[('A', 'a'), ('B', 'b'), ('C', 'c'), ('D', 'd'), ('E', 'e')] ...
hello, world!


`dict(zip(UPPER, LOWER))`: `zip` ghép từng cặp `('A','a'), ('B','b'), ...`; `dict(...)` biến list cặp thành dict. Không phải gõ tay 26 cặp. Module `string` có sẵn `string.ascii_uppercase` / `string.ascii_lowercase`.

## 9. Cách 5 — `str.maketrans` + `str.translate`

`str.maketrans(from, to)` tạo bảng dịch; `s.translate(bảng)` thay mọi ký tự theo bảng — viết bằng C nên rất nhanh. Đây là cách "Pythonic" khi muốn tự định nghĩa phép thay ký tự.

In [14]:
import string
TRANS = str.maketrans(string.ascii_uppercase, string.ascii_lowercase)

def lower_translate(s: str) -> str:
    return s.translate(TRANS)

print(string.ascii_uppercase)
print(lower_translate("Hello"), lower_translate("LOVELY"), lower_translate("@[Z]`{"))

ABCDEFGHIJKLMNOPQRSTUVWXYZ
hello lovely @[z]`{


## 10. Code nộp LeetCode

Bản tự cài bằng ASCII (đúng tinh thần bài):

In [15]:
class Solution:
    def toLowerCase(self, s: str) -> str:
        res = []
        for ch in s:
            if 'A' <= ch <= 'Z':
                res.append(chr(ord(ch) + 32))
            else:
                res.append(ch)
        return "".join(res)

sol = Solution()
for s in ["Hello", "here", "LOVELY"]:
    print(repr(sol.toLowerCase(s)))

'hello'
'here'
'lovely'


Các bản ngắn cũng được chấp nhận:

```python
class Solution:
    def toLowerCase(self, s: str) -> str:
        return s.lower()                     # dùng thư viện

class Solution:
    def toLowerCase(self, s: str) -> str:
        return "".join(chr(ord(c) | 32) if 'A' <= c <= 'Z' else c for c in s)
```

## 11. Vì sao đúng?

Hàm xử lý **từng ký tự độc lập**, nên chỉ cần mỗi ký tự được biến đổi đúng:
- Nếu `ch` là chữ hoa (`65 <= ord(ch) <= 90`): `ord(ch) + 32` nằm trong `97..122` và bằng mã của chữ thường **cùng vị trí trong bảng chữ cái** (vì cả hai dãy `A..Z`, `a..z` liên tiếp và cách nhau đúng 32) ✔.
- Nếu không phải chữ hoa: giữ nguyên, đúng yêu cầu "chỉ thay chữ cái hoa" ✔.

**Invariant vòng lặp:** sau khi xử lý `i` ký tự đầu, `''.join(res)` bằng `s[:i]` đã đổi thường. Khi `i = len(s)` ta có kết quả.

Với cách OR bit: chữ hoa có bit 5 bằng `0` (vì `65..90` = `0b1000001..0b1011010`), nên `| 32` đúng bằng `+ 32`.

In [16]:
assert all((c | 32) == c + 32 for c in range(65, 91))
assert all(format(c, "08b")[2] == "0" for c in range(65, 91))   # bit 5 (tính từ phải, 0-based) = vị trí 2 từ trái
print("Với mọi chữ hoa: c | 32 == c + 32  ✔")

Với mọi chữ hoa: c | 32 == c + 32  ✔


## 12. Độ phức tạp và so sánh

Gọi `n = len(s)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. `s.lower()` | thư viện | `O(n)` | `O(n)` | Không đạt mục tiêu học; xử lý cả Unicode |
| 2. `ord/chr + 32` | ASCII | `O(n)` | `O(n)` | **Nên nộp / nên hiểu** |
| 3. OR bit `\| 32` | bit 5 | `O(n)` | `O(n)` | Gọn, cần kiểm tra chữ hoa |
| 4. Bảng `dict` | tra cứu | `O(n)` | `O(n)` + 26 | Dễ mở rộng cho ánh xạ bất kỳ |
| 5. `translate` | bảng dịch C | `O(n)` | `O(n)` | Nhanh nhất trong Python thuần |

Space `O(n)` là chuỗi kết quả (chuỗi bất biến nên bắt buộc tạo mới).

In [17]:
import time, random
rng = random.Random(709)
big = "".join(chr(rng.randint(32, 126)) for _ in range(10**6))
for f in [lower_builtin, lower_ascii, lower_bit, lower_dict, lower_translate]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{f.__name__:16} n=10^6: {time.perf_counter() - t0:.4f}s")

lower_builtin    n=10^6: 0.0006s
lower_ascii      n=10^6: 0.0826s
lower_bit        n=10^6: 0.0844s


lower_dict       n=10^6: 0.0614s
lower_translate  n=10^6: 0.0011s


## 13. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:30} PASS -> {actual!r}")

sol = Solution()
check("mẫu 1", sol.toLowerCase("Hello"), "hello")
check("mẫu 2", sol.toLowerCase("here"), "here")
check("mẫu 3", sol.toLowerCase("LOVELY"), "lovely")
check("1 ký tự hoa", sol.toLowerCase("A"), "a")
check("1 ký tự không phải chữ", sol.toLowerCase("7"), "7")
check("biên @ A Z [", sol.toLowerCase("@AZ["), "@az[")
check("biên ` a z {", sol.toLowerCase("`az{"), "`az{")
check("khoảng trắng & dấu", sol.toLowerCase("Hi, Bob! 42?"), "hi, bob! 42?")

mẫu 1                          PASS -> 'hello'
mẫu 2                          PASS -> 'here'
mẫu 3                          PASS -> 'lovely'
1 ký tự hoa                    PASS -> 'a'
1 ký tự không phải chữ         PASS -> '7'
biên @ A Z [                   PASS -> '@az['
biên ` a z {                   PASS -> '`az{'
khoảng trắng & dấu             PASS -> 'hi, bob! 42?'


In [19]:
all_printable = "".join(chr(c) for c in range(32, 127))
funcs = [lower_ascii, lower_bit, lower_dict, lower_translate, sol.toLowerCase]
for f in funcs:
    assert f(all_printable) == all_printable.lower()
for _ in range(500):
    t = "".join(chr(rng.randint(32, 126)) for _ in range(rng.randint(1, 100)))
    exp = t.lower()
    for f in funcs:
        assert f(t) == exp
print("Toàn bộ 95 ký tự in được + 500 chuỗi ngẫu nhiên: 5 cách khớp str.lower()")

Toàn bộ 95 ký tự in được + 500 chuỗi ngẫu nhiên: 5 cách khớp str.lower()


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Cộng 32 cho **mọi** ký tự

In [20]:
def wrong_all_plus32(s):
    return "".join(chr(ord(ch) + 32) for ch in s)

print("sai:", repr(wrong_all_plus32("Hi 1!")), "| đúng:", repr(Solution().toLowerCase("Hi 1!")))

sai: 'h\x89@QA' | đúng: 'hi 1!'


Vì sao sai: `'i'` (105) thành `'\x89'`, khoảng trắng (32) thành `'@'`, `'1'` thành `'Q'`... Sửa: chỉ đổi khi `'A' <= ch <= 'Z'`.

### Lỗi 2 — Điều kiện dùng `<` thay `<=` (bỏ sót `A` hoặc `Z`)

In [21]:
def wrong_bounds(s):
    return "".join(chr(ord(c) + 32) if 'A' < c < 'Z' else c for c in s)

print("sai:", wrong_bounds("AZBY"), "| đúng:", Solution().toLowerCase("AZBY"))

sai: AZby | đúng: azby


Vì sao sai: `'A'` và `'Z'` là biên **bao gồm**. Sửa: `<=`.

### Lỗi 3 — Kiểm tra bằng `ch.isupper()` rồi tự cộng 32
`isupper()` đúng cả với chữ hoa **Unicode** (ví dụ `'Ä'`, `'Đ'`). Cộng 32 cho chúng chỉ **tình cờ** đúng với vài chữ (khối Latin-1 như `Ä→ä` cũng cách nhau 32), còn nhiều chữ khác thì sai (`'Đ'` (272) + 32 = `'İ'` chứ không phải `'đ'` (273)). Đề chỉ có ASCII nên vẫn qua, nhưng thói quen này gây lỗi với tiếng Việt. Sửa: kiểm tra đúng miền `'A' <= ch <= 'Z'`, hoặc dùng `lower()` khi xử lý Unicode thật.

In [22]:
for ch in ["Đ", "Ä"]:
    print(ch, "isupper:", ch.isupper(), "| +32 ->", repr(chr(ord(ch) + 32)), "| lower() ->", ch.lower())

Đ isupper: True | +32 -> 'İ' | lower() -> đ
Ä isupper: True | +32 -> 'ä' | lower() -> ä


### Lỗi 4 — Quên rằng chuỗi bất biến
`s[i] = ...` báo `TypeError`; còn `ch = ch.lower()` trong vòng `for ch in s` chỉ đổi biến tạm, **không** đổi `s`.

In [23]:
s = "ABC"
for ch in s:
    ch = chr(ord(ch) + 32)       # chỉ đổi biến ch
print("s vẫn là:", s)

s vẫn là: ABC


### Lỗi 5 — `return res` (list) thay vì chuỗi
Quên `''.join(res)` → trả list `['h','e',...]`, LeetCode báo sai kiểu.

## 15. Mẹo nhận dạng pattern và biến thể

- Thấy "chữ hoa/thường", "dịch ký tự", "mã hóa" → nghĩ tới **`ord/chr`** và **khoảng cách cố định** giữa các dãy ký tự.
- Công thức vị trí chữ cái: `ord(ch) - ord('a')` cho `0..25` — nền tảng cho mảng đếm 26 ô, Caesar cipher, Excel column (171).
- Đổi hoa/thường bằng bit: `| 32` (thường), `& ~32` (hoa), `^ 32` (đảo) — chỉ áp dụng cho chữ cái.

Bài liên quan: 520 Detect Capital, 917 Reverse Only Letters, 1309 Decrypt String, 2129 Capitalize the Title, 1844 Replace All Digits with Characters.

### 15.1. Biến thể A — To Upper Case

In [24]:
def to_upper(s):
    return "".join(chr(ord(c) - 32) if 'a' <= c <= 'z' else c for c in s)

print(to_upper("Hello, world!"))

HELLO, WORLD!


### 15.2. Biến thể B — Đảo hoa/thường (swapcase) bằng XOR

In [25]:
def swap_case(s):
    return "".join(chr(ord(c) ^ 32) if c.isascii() and c.isalpha() else c for c in s)

print(swap_case("Hello, World!"), "| kiểm tra:", "Hello, World!".swapcase())

hELLO, wORLD! | kiểm tra: hELLO, wORLD!


`c.isascii() and c.isalpha()`: là chữ cái **và** thuộc ASCII. Đặt `isascii()` trước để loại Unicode sớm (với Unicode, XOR 32 không có ý nghĩa).

### 15.3. Biến thể C — Detect Capital (LeetCode 520)

Một từ "dùng chữ hoa đúng" nếu: toàn hoa (`"USA"`), toàn thường (`"leetcode"`), hoặc chỉ chữ đầu hoa (`"Google"`). Ý tưởng: **đếm** số chữ hoa bằng so sánh ASCII, rồi xét 3 trường hợp.

In [26]:
def detect_capital(word: str) -> bool:
    upper = sum(1 for c in word if 'A' <= c <= 'Z')
    return upper == len(word) or upper == 0 or (upper == 1 and 'A' <= word[0] <= 'Z')

for w in ["USA", "leetcode", "Google", "FlaG", "g"]:
    print(f"{w:9} -> {detect_capital(w)}")

USA       -> True
leetcode  -> True
Google    -> True
FlaG      -> False
g         -> True


`sum(1 for c in word if ...)` đếm số phần tử thỏa điều kiện — mẫu đếm rất hay dùng. Ở `"FlaG"` có 2 chữ hoa nhưng không phải toàn hoa → `False`.

### 15.4. Bảng tóm tắt các miền ASCII hay dùng

| Nhóm | Ký tự | Mã | Kiểm tra |
|---|---|---|---|
| Chữ số | `'0'..'9'` | 48..57 | `'0' <= c <= '9'`; giá trị số: `ord(c) - ord('0')` |
| Chữ hoa | `'A'..'Z'` | 65..90 | `'A' <= c <= 'Z'`; vị trí: `ord(c) - ord('A')` |
| Chữ thường | `'a'..'z'` | 97..122 | `'a' <= c <= 'z'`; vị trí: `ord(c) - ord('a')` |
| Khoảng trắng | `' '` | 32 | `c == ' '` |

In [27]:
print([ord(c) - ord('0') for c in "2709"])
print([ord(c) - ord('a') for c in "abz"], [chr(ord('a') + i) for i in (0, 1, 25)])

[2, 7, 0, 9]
[0, 1, 25] ['a', 'b', 'z']


## 16. Bài tập tự luyện

**Bài 1 (tính tay).** `ord('G')`, rồi chữ thường của nó có mã bao nhiêu? Gán tuple `(ord_G, ord_g)` vào `ex1`.

**Bài 2 (đoán output).** `chr(ord('[') | 32)` là gì? Gán vào `ex2`.

**Bài 3 (TODO).** `ex3_to_lower(s)` tự viết bằng `ord/chr` (không dùng `lower`, `translate`).

**Bài 4 (biến thể).** `ex4_capitalize_words(s)`: viết hoa chữ cái đầu mỗi từ (từ cách nhau bởi một khoảng trắng), các chữ còn lại viết thường. Không dùng `title()`/`capitalize()`/`lower()`/`upper()`.

**Bài 5 (biến thể).** `ex5_caesar(s, k)`: dịch mỗi chữ cái (hoa/thường) đi `k` vị trí vòng tròn trong bảng chữ cái, giữ nguyên ký tự khác. Ví dụ `ex5_caesar("Zz-a", 1) == "Aa-b"`.

**Bài 6 (mini challenge).** `ex6_count_case(s)`: trả tuple `(số chữ hoa, số chữ thường, số ký tự khác)` chỉ dùng so sánh ký tự / `ord`.

In [28]:
ex1 = None   # TODO
ex2 = None   # TODO

def ex3_to_lower(s: str) -> str:
    # TODO
    pass

def ex4_capitalize_words(s: str) -> str:
    # TODO
    pass

# assert ex3_to_lower("LOVELY") == "lovely"
# assert ex4_capitalize_words("hELLO wORLD") == "Hello World"

In [29]:
def ex5_caesar(s: str, k: int) -> str:
    # TODO
    pass

def ex6_count_case(s: str):
    # TODO
    pass

# assert ex5_caesar("Zz-a", 1) == "Aa-b"
# assert ex6_count_case("Hi, Bob!") == (2, 3, 3)

## 17. Gợi ý 3 mức

- **Mức 1:** Mọi bài quy về: ký tự → số (`ord`) → biến đổi số → ký tự (`chr`).
- **Mức 2:** Bài 4: duyệt với `enumerate`; ký tự đầu từ là ký tự ở vị trí `0` hoặc ngay sau một khoảng trắng. Bài 5: đưa về `0..25` bằng `ord(c) - base` với `base = ord('a')` hoặc `ord('A')`, cộng `k`, `% 26`, cộng lại `base`.
- **Mức 3:** Bài 5:
  ```text
  base = ord('a') if 'a' <= c <= 'z' else ord('A')
  res.append(chr((ord(c) - base + k) % 26 + base))
  ```

## 18. Đáp án, template nhớ nhanh và checklist

Bài 1: `ord('G') = 71`, chữ thường `g` có mã `71 + 32 = 103`. Bài 2: `'['` là 91 = `0101 1011`, bit 5 (giá trị 32) đang là `0`, nên `91 | 32 = 123` → `'{'`. Đây là minh họa vì sao phải kiểm tra chữ hoa trước khi OR.

In [30]:
ans_ex1 = (71, 103)
ans_ex2 = '{'

def ans_ex3_to_lower(s):
    return "".join(chr(ord(c) + 32) if 'A' <= c <= 'Z' else c for c in s)

def _low(c):
    return chr(ord(c) + 32) if 'A' <= c <= 'Z' else c

def _up(c):
    return chr(ord(c) - 32) if 'a' <= c <= 'z' else c

def ans_ex4_capitalize_words(s):
    res = []
    for i, c in enumerate(s):
        if i == 0 or s[i - 1] == ' ':
            res.append(_up(c))
        else:
            res.append(_low(c))
    return "".join(res)

def ans_ex5_caesar(s, k):
    res = []
    for c in s:
        if 'a' <= c <= 'z' or 'A' <= c <= 'Z':
            base = ord('a') if 'a' <= c <= 'z' else ord('A')
            res.append(chr((ord(c) - base + k) % 26 + base))
        else:
            res.append(c)
    return "".join(res)

def ans_ex6_count_case(s):
    up = low = other = 0
    for c in s:
        if 'A' <= c <= 'Z':
            up += 1
        elif 'a' <= c <= 'z':
            low += 1
        else:
            other += 1
    return up, low, other

assert ans_ex1 == (ord('G'), ord('g'))
assert ans_ex2 == chr(ord('[') | 32)
assert ans_ex3_to_lower("LOVELY") == "lovely"
assert ans_ex4_capitalize_words("hELLO wORLD") == "Hello World"
assert ans_ex5_caesar("Zz-a", 1) == "Aa-b"
assert ans_ex5_caesar(ans_ex5_caesar("Hello, World!", 7), 26 - 7) == "Hello, World!"
assert ans_ex6_count_case("Hi, Bob!") == (2, 3, 3)
for _ in range(300):
    t = "".join(rng.choice(string.ascii_letters + "  ,.!1") for _ in range(rng.randint(1, 40)))
    assert ans_ex3_to_lower(t) == t.lower()
    u = sum(c.isupper() for c in t); l = sum(c.islower() for c in t)
    assert ans_ex6_count_case(t) == (u, l, len(t) - u - l)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


### Template nhớ nhanh

```python
class Solution:
    def toLowerCase(self, s: str) -> str:
        res = []
        for ch in s:
            if 'A' <= ch <= 'Z':
                res.append(chr(ord(ch) + 32))   # hoặc ord(ch) | 32
            else:
                res.append(ch)
        return "".join(res)
```

Câu thần chú: **"Hoa + 32 = thường; chỉ đổi khi `'A' <= ch <= 'Z'`; gom list rồi `join`."**

### Checklist trước khi submit
- [ ] Chỉ đổi ký tự trong `'A'..'Z'` (bao gồm hai biên).
- [ ] Ký tự khác (số, dấu, khoảng trắng, `@`, `[`) giữ nguyên.
- [ ] Trả **chuỗi** (`''.join`), không trả list.
- [ ] Test `"@AZ["`, `` "`az{" ``, chuỗi toàn chữ thường, chuỗi 1 ký tự.
- [ ] Time `O(n)`, Space `O(n)` cho chuỗi kết quả.